In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
data_dir = Path("data_mvg")
routes     = pd.read_csv(data_dir / 'routes.txt')
stops      = pd.read_csv(data_dir / 'stops.txt')
trips      = pd.read_csv(data_dir / 'trips.txt')
stop_times = pd.read_csv(data_dir / 'stop_times.txt')
calendar   = pd.read_csv(data_dir / 'calendar.txt')

In [3]:
tables = {
    'routes': routes,
    'stops': stops,
    'trips': trips,
    'stop_times': stop_times,
    'calendar': calendar,
}

for name, df in tables.items():
    print(f"{name}: {list(df.columns)}\n")
    print(f"shape: {df.shape}\n")

routes: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_color', 'route_text_color']

shape: (266, 8)

stops: ['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'location_type', 'parent_station']

shape: (4247, 6)

trips: ['route_id', 'service_id', 'trip_id', 'shape_id', 'trip_headsign', 'direction_id', 'block_id', 'route_direction']

shape: (53537, 8)

stop_times: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'pickup_type', 'drop_off_type', 'shape_dist_traveled']

shape: (1150460, 8)

calendar: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']

shape: (2, 10)



**Enforcing the rules of the meta.md file**

In [4]:
# Assign child stops to their parent stations

print(f"stops with a parent station: {stops['parent_station'].notnull().sum()}")
print(f"unique parent stations : {stops['parent_station'].nunique()}")

stops['station'] = stops['parent_station'].fillna(stops['stop_id'])

print(f"Unique effective stations: {stops['station'].nunique()} should equal parent stops")


# Check the routes table for the number of unique route types
print(f"Unique route_id: {routes['route_id'].nunique()}")
print(f"Unique route_short_name: {routes['route_short_name'].nunique()}\n")

# Route types distribution 
print(routes['route_desc'].value_counts())

stops with a parent station: 3026
unique parent stations : 1221
Unique effective stations: 1221 should equal parent stops
Unique route_id: 266
Unique route_short_name: 127

route_desc
Stadtbus                 101
Metrobus                  66
Tram                      59
U-Bahn                    27
ExpressBus                 9
Schienenersatzverkehr      4
Name: count, dtype: int64


In [ ]:
# Checks for clean data from the start so we won't have to deal with it later

# Times format
print("Sample times:")
print(stop_times[['arrival_time', 'departure_time']].head(10))

print(f"Max arrival_time: {stop_times['arrival_time'].max()}\n") # a number >24 means probably a time after midnight when the bus arrives at the stop while the bus starts its trip the day before
print(stop_times[['arrival_time', 'departure_time']].dtypes,"\n") # might need to convert to datetime later 

# Distances 
print(f"shape_dist_traveled nulls: {stop_times['shape_dist_traveled'].isna().sum()}\n")

#lat/lon completeness:
print(f"stop_lat nulls: {stops['stop_lat'].isna().sum()}")
print(f"stop_lon nulls: {stops['stop_lon'].isna().sum()}\n")

# checking if stop times reference child stops 
child_stops = stops[stops['parent_station'].notna()]['stop_id']
refs_to_children = stop_times['stop_id'].isin(child_stops).sum()
print(f"stop_times rows referencing child stops: {refs_to_children} / {len(stop_times)}")



Sample times:
  arrival_time departure_time
0     03:19:50       03:19:50
1     03:20:50       03:21:10
2     03:22:10       03:22:30
3     03:23:30       03:23:50
4     03:25:00       03:26:20
5     03:27:40       03:28:00
6     03:29:00       03:29:20
7     03:30:30       03:37:20
8     03:38:30       03:39:20
9     03:40:20       03:40:40
Max arrival_time: 27:43:40

arrival_time      str
departure_time    str
dtype: object 

shape_dist_traveled nulls: 0

stop_lat nulls: 0
stop_lon nulls: 0

stop_times rows referencing child stops: 1150460 / 1150460


In [19]:
# Map the child stops to their parent stations in the stop_times table

stop_id_to_station = stops.set_index('stop_id')['station']
stop_times['station'] = stop_times['stop_id'].map(stop_id_to_station)
print(f"Unmapped stations: {stop_times['station'].isna().sum()}")

Unmapped stations: 0
